# Why We Were Named a 2025 Gartner Cool Vendor in LLM Observability

Flying blind in production AI is a recipe for disaster. If you can't trace a specific user's request, debug a structured tool-call failure, or track token costs in real-time, you are not ready for scale. 

In this cookbook, we'll explore Portkey's deep tracing and observability layer—the unified control plane that processes 500B+ tokens and earned us Gartner's recognition as a 2025 Cool Vendor.

## What You'll Learn
- Appending custom metadata (users, environments, session IDs) to requests
- Capturing structured tool calls and complex LLM interactions
- Viewing real-time request logs and traces in the Portkey Dashboard

## Prerequisites
- Python 3.9+
- Portkey API Key ([get one free](https://app.portkey.ai))
- OpenAI API Key

**Time to complete**: ~5 minutes


## Step 1: Installation

First, install the Portkey SDK if you haven't already.

In [ ]:
# Install Portkey AI SDK
!pip install -q portkey-ai

## Step 2: Setup & Initialization

Initialize the Portkey client. We will configure default metadata so every API call made with this client is automatically tagged. This is incredibly powerful for tracking costs per tenant or debugging specific user sessions.

In [ ]:
import os
from portkey_ai import Portkey

# Set your keys here (we recommend using environment variables)
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY", "your-portkey-api-key")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")

# Initialize Portkey with custom metadata
client = Portkey(
    api_key=PORTKEY_API_KEY,
    provider="openai",
    Authorization=OPENAI_API_KEY,
    # These trace IDs will appear in your Portkey dashboard for every request
    trace_id="onboarding_flow_v2",
    metadata={
        "_environment": "production",
        "_user": "analytics_team_1",
        "feature": "data_extraction"
    }
)

print("✅ Portkey client initialized with tracing metadata!")

## Step 3: Deep Tracing for Structured Tool Calls

One of the hardest things to debug in modern LLM applications is tool calling (function calling). Portkey automatically tracks and visualizes the exact JSON payload the model outputs, and pairs it with the execution result.

In [ ]:
# Let's simulate a complex request with tool definitions
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "The city and state"},
                },
                "required": ["location"],
            },
        }
    }
]

try:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "What's the weather like in San Francisco?"}],
        tools=tools,
        tool_choice="auto"
    )
    
    print("✅ Request successful! Check your Portkey dashboard to see the deep trace.\n")
    
    if response.choices[0].message.tool_calls:
        print("The LLM decided to call a tool:")
        for tool_call in response.choices[0].message.tool_calls:
            print(f"- Function Name: {tool_call.function.name}")
            print(f"- Arguments: {tool_call.function.arguments}")
            
except Exception as e:
    print(f"❌ Error: {e}")

## 🔄 Switch Providers in One Line

Need to compare how Anthropic handles tool calls vs OpenAI? Just change the provider at initialization. Your metadata and tracing capabilities remain exactly the same.

In [ ]:
try:
    # Switch to Anthropic with the exact same tracing metadata
    anthropic_client = Portkey(
        api_key=PORTKEY_API_KEY,
        provider="anthropic",
        Authorization=os.getenv("ANTHROPIC_API_KEY", "your-anthropic-key"),
        trace_id="onboarding_flow_v2",
        metadata={
            "_environment": "production",
            "_user": "analytics_team_1",
            "feature": "data_extraction"
        }
    )
    
    print("✨ Ready to natively trace Anthropic requests in the unified format!")
except Exception as e:
    print(e)

## 🎯 Key Takeaways

1. **Unified Observability**: Whether you call OpenAI, Anthropic, or Azure, the logs all flow into a standardized, searchable format in your Portkey dashboard.
2. **Granular Cost Tracking**: By tagging requests with `_user` or `_organization` metadata, you can see exactly who is driving your LLM costs.
3. **Tool-Call Visibility**: Unlike raw logs, Portkey structures and highlights function calls, making it instantly clear when an LLM hallucinated a parameter.

## Common Gotchas
- Metadata keys starting with `_` (like `_user` and `_environment`) are treated as special reserved keys by Portkey for advanced filtering in the dashboard.
- The `trace_id` groups multiple requests together into a single visual span in the logs (great for Agentic workflows!).

## 🚀 Next Steps

- **Analyze Logs**: Head to [app.portkey.ai](https://app.portkey.ai) to view your traces.
- **Export with OpenTelemetry**: [Stream logs to Datadog, New Relic, or Splunk](https://portkey.ai/docs/integrations/llm-observability)
- **Cost Optimization**: Use your new visibility to track exactly where tokens are being wasted.

---

⭐ **Star our gateway**: [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway)

📚 **Full docs**: [portkey.ai/docs](https://portkey.ai/docs/)

🐦 **Follow us**: [@PortkeyAI](https://x.com/PortkeyAI)
